# ARDL tutorial

This notebook demonstrates the standard autoregressive distributed-lag API in `Ts.TsModels`. ARDL models target-variable lags and finite input lags directly; it is distinct from the rational transfer-function support exposed through `SARIMAX(..., distributed_lags=...)`.

## Goal and setup

Fit one manually specified ARDL, select lags automatically with BIC, and produce a conditional forecast from an explicit future input scenario.

In [ ]:
import pandas as pd
from statsmodels.datasets.danish_data import load

from Ts.TsModels import ARDL, AutoARDL

data = load().data
response = data["lrm"]
inputs = data[["lry", "ibo"]]
response.shape, inputs.shape

## Manual ARDL

The target uses lags 1 and 2. `lry` uses current and first-lag values, while `ibo` uses current through lag 2.

In [ ]:
manual = ARDL(
    response,
    lags=[1, 2],
    exog=inputs,
    order={"lry": [0, 1], "ibo": [0, 1, 2]},
    trend="c",
    missing="raise",
).fit(cov_type="HC1")

print(manual.summary())

In [ ]:
{
    "target_lags": manual.ar_lags,
    "input_lags": manual.distributed_lags,
    "stable_target_ar": manual.is_stationary,
    "effective_nobs": manual.effective_nobs,
}

## Automatic lag selection

Hierarchical selection preserves nested lag structures and is the default. Global subset search is available with `search_method="global"`, but its search space grows rapidly.

In [ ]:
automatic = AutoARDL(
    response,
    maxlag=3,
    exog=inputs,
    maxorder={"lry": 2, "ibo": 2},
    criterion="bic",
    search_method="hierarchical",
    missing="raise",
).fit()

automatic.criterion_table.head()

In [ ]:
{
    "selected_target_lags": automatic.ar_lags,
    "selected_input_lags": automatic.distributed_lags,
    "bic": automatic.bic,
}

## Conditional forecast

ARDL forecasts require every future input value. The values below are an explicit teaching scenario copied from the last four historical rows; they are not observed future data or a deployment forecast.

In [ ]:
future_index = pd.date_range(
    start=response.index[-1], periods=5, freq=response.index.freq
)[1:]
future_scenario = inputs.iloc[-4:].copy()
future_scenario.index = future_index
forecast = automatic.predict(
    start=len(response),
    end=len(response) + 3,
    future_exog=future_scenario,
)
pd.DataFrame(
    {"mean": forecast.mean, "lower": forecast.lower, "upper": forecast.upper},
    index=future_index,
)

## Checks and next steps

Before interpreting an ARDL, inspect the selected lag structure, target-AR stability, residual diagnostics, and sensitivity to the maximum lag bounds. Cointegration bounds tests and UECM reparameterization are deliberately outside this API.